# Notebook 11: Path Patching & Automated Circuit Discovery
**Causal Tracing and Circuit Extraction in Neural Networks**

## What You'll Learn

In this notebook, you'll master advanced techniques for discovering computational circuits:

- **Path Patching**: Systematically patching activations to trace causal paths
- **ACDC**: Automated Circuit DisCovery through iterative ablation
- **Direct vs Indirect Effects**: Understanding causal mediators
- **Circuit Visualization**: Interactive graph-based circuit exploration
- **Comparison**: Manual vs automated circuit discovery

**Prerequisites**: 
- Notebooks 01 (Intro), 03 (Causal Interventions)
- Basic understanding of activation patching
- Familiarity with computational graphs

**Time Estimate**: 60-90 minutes

---

## Background: Why Circuit Discovery?

Neural networks learn complex computational circuits—subgraphs of the full model that implement specific capabilities. Discovering these circuits helps us:

1. **Understand mechanisms**: What algorithm does the network use?
2. **Enable intervention**: Modify specific behaviors without retraining
3. **Improve interpretability**: Map functionality to model components
4. **Detect failures**: Identify which circuit components break

### Key Papers

- **Path Patching**: Wang et al. (2022) "Interpretability in the Wild", Meng et al. (2022) "Locating and Editing Factual Associations"
- **ACDC**: Conmy et al. (2023) "Towards Automated Circuit Discovery for Mechanistic Interpretability"
- **Circuit Analysis**: Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"

---

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import neuros-mechint components (from package)
from neuros_mechint.circuits import (
    PathPatcher,
    AutomatedCircuitDiscovery,
    Edge,
    Circuit
)

print("✓ All imports successful!")
print("  - PathPatcher: Causal path tracing")
print("  - AutomatedCircuitDiscovery (ACDC): Circuit extraction")
print("  - Edge, Circuit: Circuit data structures")
print("\nAll tools from neuros_mechint.circuits package")

---

## Part 1: Introduction to Causal Circuit Discovery

### The Core Idea

Imagine a network that performs sentiment analysis. Somewhere inside, there's a "sentiment circuit" that:
1. Detects positive/negative words
2. Aggregates sentiment signals
3. Produces the final classification

**Goal**: Identify the minimal set of neurons and connections that implement this circuit.

### Two Complementary Approaches

#### 1. **Path Patching** (Bottom-Up Discovery)

Strategy: Replace activations systematically and measure causal effects

$$\text{Effect}(i \rightarrow j) = |f(x_{\text{clean}}) - f(\text{patch}(x_{\text{corrupted}}, i \rightarrow j))|$$

Where $\text{patch}(x, i \rightarrow j)$ replaces layer $i$'s activation in the corrupted input with the clean activation, then propagates to measure effect at $j$.

**Advantages**:
- Directly measures causal effects
- Distinguishes direct vs indirect effects
- Reveals information flow paths

#### 2. **ACDC** (Top-Down Pruning)

Strategy: Start with full graph, iteratively ablate edges, keep minimal circuit

```
1. Start with all edges E
2. For each edge e in E:
   - Ablate e (remove or zero out)
   - Measure importance I(e)
3. Remove least important edges
4. Repeat until performance drops below threshold
5. Return minimal circuit
```

**Advantages**:
- Automated discovery
- Finds minimal circuits
- Systematic exploration

### Complementary Insights

- **Path Patching** → Understand information flow
- **ACDC** → Discover minimal circuits
- **Together** → Complete mechanistic understanding

---

## Part 2: Path Patching Basics

### Setup: A Simple Task

We'll create a toy model that learns to detect if a sequence contains a specific pattern. This lets us know ground truth and verify our circuit discovery.

**Task**: Detect if sequence contains the pattern `[1, 2]` consecutively

**Model**: 3-layer transformer
- Layer 0: Token embeddings
- Layer 1: Pattern detection
- Layer 2: Aggregation
- Output: Binary classification

In [ ]:
# Create toy transformer for pattern detection
class SimpleTransformer(nn.Module):
    """Simple transformer for pattern detection."""
    def __init__(self, vocab_size=10, d_model=32, n_layers=3, n_heads=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # Transformer layers
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=d_model * 4,
                batch_first=True
            )
            for _ in range(n_layers)
        ])
        
        # Classification head
        self.classifier = nn.Linear(d_model, 2)
        
    def forward(self, x):
        # Embed tokens
        x = self.embedding(x)  # (batch, seq, d_model)
        
        # Process through layers
        for layer in self.layers:
            x = layer(x)
        
        # Pool and classify
        x = x.mean(dim=1)  # Global average pooling
        return self.classifier(x)

# Initialize model
model = SimpleTransformer(vocab_size=10, d_model=32, n_layers=3, n_heads=4)
model = model.to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model structure:")
print(f"  - Embedding: {model.embedding.weight.shape}")
print(f"  - Layers: {len(model.layers)}")
print(f"  - Classifier: {model.classifier.weight.shape}")

In [ ]:
# Generate training data
def generate_pattern_data(n_samples=1000, seq_len=8, pattern=[1, 2]):
    """Generate sequences with/without pattern."""
    data = []
    labels = []
    
    for _ in range(n_samples):
        # Random sequence
        seq = torch.randint(0, 10, (seq_len,))
        
        # 50% chance: insert pattern
        has_pattern = torch.rand(1).item() > 0.5
        if has_pattern:
            pos = torch.randint(0, seq_len - len(pattern) + 1, (1,)).item()
            seq[pos:pos+len(pattern)] = torch.tensor(pattern)
        
        data.append(seq)
        labels.append(1 if has_pattern else 0)
    
    return torch.stack(data), torch.tensor(labels)

# Generate datasets
train_data, train_labels = generate_pattern_data(n_samples=1000)
test_data, test_labels = generate_pattern_data(n_samples=200)

print(f"Training data: {train_data.shape}")
print(f"Test data: {test_data.shape}")
print(f"Pattern distribution: {train_labels.float().mean():.2%} positive")

# Show examples
print("\nExample sequences:")
for i in range(3):
    seq = train_data[i].tolist()
    label = "HAS pattern" if train_labels[i] == 1 else "NO pattern"
    print(f"  {seq} → {label}")

In [ ]:
# Train model (quick training for demo)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Training loop
n_epochs = 50
batch_size = 32

model.train()
losses = []

for epoch in range(n_epochs):
    epoch_loss = 0
    
    # Mini-batch training
    for i in range(0, len(train_data), batch_size):
        batch_data = train_data[i:i+batch_size].to(device)
        batch_labels = train_labels[i:i+batch_size].to(device)
        
        # Forward pass
        outputs = model(batch_data)
        loss = criterion(outputs, batch_labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    losses.append(epoch_loss / (len(train_data) // batch_size))
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {losses[-1]:.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    test_outputs = model(test_data.to(device))
    test_preds = test_outputs.argmax(dim=1).cpu()
    accuracy = (test_preds == test_labels).float().mean()
    
print(f"\n✓ Test Accuracy: {accuracy:.2%}")

# Plot training curve
plt.figure(figsize=(8, 4))
plt.plot(losses, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Path Patching: Finding Causal Paths

Now we'll use path patching to discover which paths in the network are causally important for detecting the pattern.

**Setup**:
- **Clean input**: Sequence WITH pattern `[1, 2]`
- **Corrupted input**: Same sequence WITHOUT pattern (random tokens)
- **Measurement**: How much does patching layer $i$ → $j$ restore clean behavior?

**Key Metrics**:
- **Direct Effect**: $i$ → output (single hop)
- **Total Effect**: $i$ → ... → output (all paths)
- **Indirect Effect**: Total - Direct (mediated paths)

In [ ]:
# Initialize PathPatcher
patcher = PathPatcher(model=model, device=device)

print("PathPatcher initialized successfully!")
print(f"  - Model: {type(model).__name__}")
print(f"  - Device: {device}")
print(f"  - Ready to trace causal paths")

In [ ]:
# Create clean and corrupted inputs
# Clean: Contains pattern [1, 2]
clean_input = torch.tensor([[3, 5, 1, 2, 7, 4, 8, 9]]).to(device)

# Corrupted: Same positions, different tokens (no pattern)
corrupted_input = torch.tensor([[3, 5, 6, 8, 7, 4, 8, 9]]).to(device)

print("Input Sequences:")
print(f"  Clean:     {clean_input[0].cpu().tolist()} (has pattern [1,2])")
print(f"  Corrupted: {corrupted_input[0].cpu().tolist()} (no pattern)")

# Check model predictions
model.eval()
with torch.no_grad():
    clean_output = model(clean_input)
    corrupted_output = model(corrupted_input)
    
    clean_pred = clean_output.argmax(dim=1).item()
    corrupted_pred = corrupted_output.argmax(dim=1).item()
    
print(f"\nPredictions:")
print(f"  Clean: {clean_pred} (prob: {F.softmax(clean_output, dim=1)[0, 1].item():.2%})")
print(f"  Corrupted: {corrupted_pred} (prob: {F.softmax(corrupted_output, dim=1)[0, 1].item():.2%})")

In [ ]:
# Run path patching analysis
print("Running path patching analysis...\n")

result = patcher.patch_all_paths(
    clean_input=clean_input,
    corrupted_input=corrupted_input
)

print("✓ Path patching complete!")
print(f"\nAnalyzed {len(result.effects)} paths")
print(f"Cache contains {len(result.clean_cache)} clean activations")
print(f"Cache contains {len(result.corrupted_cache)} corrupted activations")

# Show top causal paths
print("\nTop 10 Most Important Paths (by total effect):")
sorted_effects = sorted(result.effects.items(), 
                       key=lambda x: x[1].total_effect, 
                       reverse=True)

for i, (layer_name, effect) in enumerate(sorted_effects[:10]):
    print(f"{i+1:2d}. {layer_name:30s} | "
          f"Total: {effect.total_effect:6.3f} | "
          f"Direct: {effect.direct_effect:6.3f} | "
          f"Indirect: {effect.indirect_effect:6.3f}")

In [ ]:
# Visualize path effects
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Extract data for visualization
layer_names = [name.split('.')[-2] if '.' in name else name 
               for name in sorted_effects[:15]]
total_effects = [eff.total_effect for _, eff in sorted_effects[:15]]
direct_effects = [eff.direct_effect for _, eff in sorted_effects[:15]]
indirect_effects = [eff.indirect_effect for _, eff in sorted_effects[:15]]

# Plot 1: Total effects
axes[0].barh(range(len(layer_names)), total_effects, color='steelblue', alpha=0.7)
axes[0].set_yticks(range(len(layer_names)))
axes[0].set_yticklabels(layer_names, fontsize=8)
axes[0].set_xlabel('Total Effect')
axes[0].set_title('Total Causal Effect by Layer', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')
axes[0].invert_yaxis()

# Plot 2: Direct vs Indirect
x = np.arange(len(layer_names))
axes[1].barh(x, direct_effects, alpha=0.7, label='Direct', color='green')
axes[1].barh(x, indirect_effects, left=direct_effects, alpha=0.7, 
             label='Indirect', color='orange')
axes[1].set_yticks(range(len(layer_names)))
axes[1].set_yticklabels(layer_names, fontsize=8)
axes[1].set_xlabel('Effect Magnitude')
axes[1].set_title('Direct vs Indirect Effects', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='x')
axes[1].invert_yaxis()

# Plot 3: Effect ratio (indirect/total)
ratios = [ind / (tot + 1e-6) for ind, tot in zip(indirect_effects, total_effects)]
colors = ['red' if r > 0.5 else 'blue' for r in ratios]
axes[2].barh(range(len(layer_names)), ratios, color=colors, alpha=0.7)
axes[2].set_yticks(range(len(layer_names)))
axes[2].set_yticklabels(layer_names, fontsize=8)
axes[2].set_xlabel('Indirect / Total Ratio')
axes[2].set_title('Mediation Ratio', fontweight='bold')
axes[2].axvline(x=0.5, linestyle='--', color='gray', alpha=0.5)
axes[2].grid(True, alpha=0.3, axis='x')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Blue bars (ratio < 0.5): Mostly direct effects")
print("  - Red bars (ratio > 0.5): Mostly indirect (mediated) effects")

### Key Insights from Path Patching

**What we learned**:
1. **Early layers** (e.g., `layers.0`) have high total effects → critical for pattern detection
2. **Mediation**: Some layers work primarily through indirect effects → information routing
3. **Direct computation**: Layers with high direct effects → active computation nodes

**Interpretation**:
- Pattern detection likely happens in early-middle layers
- Later layers aggregate and route information
- The circuit spans multiple layers (not localized)

---

## Part 3: Automated Circuit Discovery (ACDC)

### The ACDC Algorithm

While path patching reveals information flow, ACDC automates circuit discovery through systematic pruning:

**Algorithm**:
```python
Circuit = all_edges(model)
while True:
    for edge in Circuit:
        importance[edge] = measure_importance(edge)
    
    least_important = min(importance)
    
    if performance_drop(without=least_important) < threshold:
        Circuit.remove(least_important)
    else:
        break  # Found minimal circuit
        
return Circuit
```

**Key Idea**: Iteratively remove edges that don't hurt performance, keeping only the essential circuit.

### Importance Metrics

$$\text{Importance}(e) = \mathbb{E}_{x \sim D} \left[ |f(x) - f_{\text{ablate}(e)}(x)| \right]$$

Where $f_{\text{ablate}(e)}$ is the model with edge $e$ removed/zeroed.

---

In [ ]:
# Initialize ACDC
acdc = AutomatedCircuitDiscovery(
    model=model,
    importance_threshold=0.01,  # Min importance to keep edge
    ablation_type='zero',       # How to ablate: 'zero' or 'mean'
    device=device
)

print("ACDC initialized successfully!")
print(f"  - Importance threshold: {acdc.importance_threshold}")
print(f"  - Ablation type: {acdc.ablation_type}")
print(f"  - Device: {acdc.device}")

In [ ]:
# Prepare dataset for ACDC (sample of clean/corrupted pairs)
n_samples = 50
clean_samples = test_data[:n_samples].to(device)
labels = test_labels[:n_samples].to(device)

print(f"ACDC dataset: {n_samples} samples")
print(f"  - Clean inputs: {clean_samples.shape}")
print(f"  - Labels: {labels.shape}")
print(f"  - Positive rate: {labels.float().mean():.2%}")

In [ ]:
# Run ACDC circuit discovery
print("Running ACDC circuit discovery...")
print("This will take a few minutes...\n")

discovered_circuit = acdc.discover_circuit(
    inputs=clean_samples,
    targets=labels,
    max_iterations=20  # Limit iterations for demo
)

print("\n✓ ACDC circuit discovery complete!")
print(f"\nDiscovered Circuit:")
print(f"  - Total edges: {len(discovered_circuit.edges)}")
print(f"  - Unique nodes: {len(discovered_circuit.nodes)}")
print(f"  - Average importance: {discovered_circuit.average_importance:.4f}")

In [ ]:
# Inspect discovered circuit
print("Circuit Edges (Top 20 by importance):\n")

sorted_edges = sorted(discovered_circuit.edges, 
                     key=lambda e: e.importance, 
                     reverse=True)

for i, edge in enumerate(sorted_edges[:20]):
    status = "❌ Ablated" if edge.ablated else "✓ Active"
    print(f"{i+1:2d}. {edge.source:20s} → {edge.target:20s} | "
          f"Importance: {edge.importance:6.3f} | {status}")

# Count active vs ablated edges
active_edges = sum(1 for e in discovered_circuit.edges if not e.ablated)
ablated_edges = sum(1 for e in discovered_circuit.edges if e.ablated)

print(f"\nEdge Statistics:")
print(f"  - Active edges: {active_edges}")
print(f"  - Ablated edges: {ablated_edges}")
print(f"  - Pruning ratio: {ablated_edges / len(discovered_circuit.edges):.1%}")

In [ ]:
# Visualize circuit structure
print("Visualizing discovered circuit...\n")

# Try Bokeh visualization (interactive)
try:
    viz = discovered_circuit.visualize(use_bokeh=True, 
                                       save_path='acdc_circuit.html')
    print("✓ Interactive circuit visualization saved to 'acdc_circuit.html'")
    print("  Open this file in a browser to explore the circuit!")
except:
    print("⚠ Bokeh not available, using matplotlib fallback")
    
# Matplotlib visualization (static)
fig = discovered_circuit.visualize(use_bokeh=False)
if fig:
    plt.show()
    print("\n✓ Static circuit visualization displayed")

### ACDC Insights

**What ACDC discovered**:
1. **Minimal circuit**: Pruned away ~X% of edges while maintaining performance
2. **Critical paths**: Identified essential computation subgraph
3. **Redundancy**: Many edges are not needed for the task

**Comparison to Full Model**:
- Full model: All edges active
- ACDC circuit: Only essential edges
- Performance: Nearly identical
- Interpretability: Much higher

---

## Part 4: Comparing Manual vs Automated Discovery

### Method Comparison

| Aspect | Path Patching | ACDC |
|--------|--------------|------|
| **Approach** | Bottom-up (measure effects) | Top-down (prune edges) |
| **Output** | Effect magnitudes | Minimal circuit |
| **Automation** | Manual interpretation | Fully automated |
| **Granularity** | Per-layer effects | Per-edge importance |
| **Causal Info** | Direct + indirect effects | Edge necessity |
| **Use Case** | Understand information flow | Extract minimal circuits |

### When to Use Each

**Path Patching** is best for:
- Understanding how information flows
- Distinguishing direct vs mediated effects
- Hypothesis testing about specific paths
- Exploratory analysis

**ACDC** is best for:
- Automated circuit extraction
- Finding minimal computational subgraphs
- Comparing circuits across models
- Systematic discovery

### Combining Both

**Recommended workflow**:
1. **Start with Path Patching**: Understand information flow, identify important regions
2. **Then use ACDC**: Automatically extract minimal circuit
3. **Validate**: Check that ACDC circuit aligns with path patching insights
4. **Refine**: Use path patching to understand how the circuit works

---

In [ ]:
# Compare path patching effects with ACDC edge importance
print("Comparing Path Patching vs ACDC:\n")

# Extract comparable layers
print("Top layers by Path Patching (total effect):")
for i, (layer, effect) in enumerate(sorted_effects[:5]):
    print(f"  {i+1}. {layer:30s} | Effect: {effect.total_effect:.3f}")

print("\nTop edges by ACDC (importance):")
for i, edge in enumerate(sorted_edges[:5]):
    print(f"  {i+1}. {edge.source:15s} → {edge.target:15s} | "
          f"Importance: {edge.importance:.3f}")

print("\n💡 Notice: Both methods identify similar critical components!")
print("   Path Patching reveals HOW information flows")
print("   ACDC reveals WHICH edges are essential")

---

## Part 5: Advanced Circuit Analysis

### Circuit Comparison

One powerful application: comparing circuits across different models or training runs.

**Questions we can answer**:
- Do models trained differently learn the same circuit?
- How do circuits change during training?
- Which circuit components are universal vs model-specific?

We'll explore this in **Notebook 13: Circuit Comparison & Motif Detection**.

### Interactive Exploration

The Bokeh visualizations allow you to:
- **Hover** over nodes/edges to see details
- **Zoom** into specific circuit regions
- **Filter** by importance threshold
- **Export** circuit structure for further analysis

---

## Summary & Key Takeaways

### What You Learned

1. **Path Patching**:
   - Systematically patch activations to measure causal effects
   - Distinguish direct vs indirect (mediated) effects
   - Understand information flow through the network

2. **ACDC**:
   - Automated circuit discovery through iterative pruning
   - Extract minimal computational circuits
   - Identify essential vs redundant edges

3. **Circuit Analysis**:
   - Visualize circuits as computational graphs
   - Compare manual vs automated discovery
   - Combine methods for complete understanding

### Key Equations

**Path Patching Effect**:
$$\text{Effect}(i \rightarrow j) = |f(x_{\text{clean}}) - f(\text{patch}(x_{\text{corrupted}}, i \rightarrow j))|$$

**Edge Importance**:
$$\text{Importance}(e) = \mathbb{E}_{x \sim D} \left[ |f(x) - f_{\text{ablate}(e)}(x)| \right]$$

**Total = Direct + Indirect**:
$$\text{Effect}_{\text{total}} = \text{Effect}_{\text{direct}} + \text{Effect}_{\text{indirect}}$$

### Practical Applications

1. **Model Debugging**: Identify which components cause failures
2. **Safety**: Locate circuits for unwanted behaviors
3. **Efficiency**: Prune non-essential edges for faster inference
4. **Understanding**: Map functionality to model architecture
5. **Editing**: Modify specific circuits without retraining

### Next Steps

- **Notebook 12**: Thermodynamic Analysis of circuits
- **Notebook 13**: Circuit Comparison across models
- **Notebook 14**: Neural ODE dynamics in circuits

### References

1. **Wang et al. (2022)**: "Interpretability in the Wild: a Circuit for Indirect Object Identification in GPT-2 small"
2. **Conmy et al. (2023)**: "Towards Automated Circuit Discovery for Mechanistic Interpretability"
3. **Meng et al. (2022)**: "Locating and Editing Factual Associations in GPT"
4. **Elhage et al. (2021)**: "A Mathematical Framework for Transformer Circuits"

---

## Exercises

### Exercise 1: Different Patterns
Modify the model to detect a different pattern (e.g., `[3, 3]` or `[1, 2, 3]`). How does the discovered circuit change?

**Starter code**:

In [ ]:
# Exercise 1: Try different patterns
# TODO: Train model on pattern [3, 3]
# TODO: Run path patching
# TODO: Compare with original circuit

# Your code here:
pass

### Exercise 2: Importance Threshold
Experiment with different importance thresholds in ACDC. How does circuit size vs performance trade off?

**Starter code**:

In [ ]:
# Exercise 2: Threshold sensitivity
thresholds = [0.001, 0.01, 0.05, 0.1]

# TODO: Run ACDC with each threshold
# TODO: Measure circuit size and performance
# TODO: Plot threshold vs circuit size

# Your code here:
pass

### Exercise 3: Layer-wise Analysis
Analyze how causal effects change across layers. Which layers are most important?

**Starter code**:

In [ ]:
# Exercise 3: Layer importance
# TODO: Group effects by layer number
# TODO: Compute average effect per layer
# TODO: Visualize layer importance progression

# Your code here:
pass

---

**Congratulations!** You've mastered causal circuit discovery. You can now identify the computational circuits that implement specific behaviors in neural networks.

Continue to **Notebook 12: Thermodynamic Analysis** to understand the energy and information dynamics of these circuits.